## 1. Importación de Librerías e identificación de las fuentes de datos

In [1]:
#rutas de trabajo
from pathlib import Path
import os
import pandas as pd
import numpy as np
import json
import sqlite3

BASE_DIR = Path("../")
DATOS = BASE_DIR / "datos"

DATOS.mkdir(parents=True, exist_ok=True)
print("Carpeta de datos:", DATOS)

Carpeta de datos: ..\datos


## 2. Base de datos SQL con al menos una tabla relacionada

`datos/fuentes/hr_analytics.sql` crea 3 tablas relacionadas por llave foránea:
`departamentos` → `puestos` → `empleados`. La siguiente celda corre ese script
y genera la base de datos SQLite (`hr_analytics.db`).

In [6]:
ruta_sql_script = BASE_DIR / "datos" / "fuentes" / "hr_analytics.sql"
ruta_db = BASE_DIR / "datos" / "fuentes" / "hr_analytics.db"

if ruta_db.exists():
    ruta_db.unlink()  # por si ya existe de una corrida anterior

with open(ruta_sql_script, encoding="utf-8-sig") as f:
    script_sql = f.read()

conexion = sqlite3.connect(ruta_db)
conexion.executescript(script_sql)
conexion.commit()
conexion.close()

print(f"Base de datos SQL creada en: {ruta_db}")

Base de datos SQL creada en: ..\datos\fuentes\hr_analytics.db


## 3. Leer las tres fuentes desde Python

In [8]:
def leer_csv(ruta):
    df = pd.read_csv(ruta, encoding="utf-8-sig")
    df["empleado_id"] = df["empleado_id"].astype(str).str.strip()
    df["fecha_evaluacion"] = pd.to_datetime(df["fecha_evaluacion"], errors="coerce")
    return df


def leer_api_json(ruta):
    with open(ruta, encoding="utf-8-sig") as f:
        payload = json.load(f)
    registros = payload.get("data", [])
    df = pd.json_normalize(registros, sep="_")  # aplana el sub-objeto "respuestas"
    df["empleado_id"] = df["empleado_id"].astype(str).str.strip()
    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
    df = df.rename(columns={"fecha": "fecha_encuesta"})
    return df


def leer_sql_db(ruta_db):
    conexion = sqlite3.connect(ruta_db)
    query = """
        SELECT
            e.empleado_id, e.nombre_completo, e.genero, e.edad, e.fecha_ingreso,
            e.salario_mensual, e.modalidad, e.estado_laboral, e.fecha_salida,
            p.nombre AS puesto, p.nivel AS nivel_puesto,
            d.nombre AS departamento, d.ubicacion AS ubicacion_departamento
        FROM empleados e
        INNER JOIN puestos p ON e.puesto_id = p.puesto_id
        INNER JOIN departamentos d ON p.departamento_id = d.departamento_id
    """
    df = pd.read_sql_query(query, conexion)
    conexion.close()
    df["empleado_id"] = df["empleado_id"].astype(str).str.strip()
    df["fecha_ingreso"] = pd.to_datetime(df["fecha_ingreso"], errors="coerce")
    df["fecha_salida"] = pd.to_datetime(df["fecha_salida"], errors="coerce")
    return df


ruta_csv = BASE_DIR / "datos" / "fuentes" / "desempeno_historico.csv"
ruta_json = BASE_DIR / "datos" / "fuentes" / "encuestas_clima_api.json"

df_desempeno = leer_csv(ruta_csv)
df_encuestas = leer_api_json(ruta_json)
df_empleados = leer_sql_db(ruta_db)

print(f"Desempeño: {df_desempeno.shape}")
print(f"Encuestas: {df_encuestas.shape}")
print(f"Empleados: {df_empleados.shape}")

Desempeño: (168, 25)
Encuestas: (42, 21)
Empleados: (40, 13)


In [9]:
df_desempeno.head()

,registro_id,empleado_id,periodo,fecha_evaluacion,productividad_pct,calidad_trabajo_pct,cumplimiento_objetivos_pct,asistencia_pct,horas_capacitacion,horas_extra,...,pais,departamento_geo,municipio,distrito,direccion_referencia,latitud,longitud,zona_geografica,fuente_registro,fecha_extraccion
0,1,EMP001,2025-Q3,2025-09-30,76.2,72.1,68.6,97.6,12,9,...,El Salvador,San Salvador,San Salvador Centro,San Salvador,"Centro de San Salvador, San Salvador, El Salvador",13.681363,-89.226416,Centro,Sistema de desempeño,2026-07-26
1,2,EMP001,2025-Q4,2025-12-31,56.2,76.7,71.5,94.1,11,12,...,El Salvador,San Salvador,San Salvador Centro,San Salvador,"Centro de San Salvador, San Salvador, El Salvador",13.681363,-89.226416,Centro,Sistema de desempeño,2026-07-26
2,3,EMP001,2026-Q1,2026-03-31,75.1,67.9,76.0,92.8,10,19,...,El Salvador,San Salvador,San Salvador Centro,San Salvador,"Centro de San Salvador, San Salvador, El Salvador",13.681363,-89.226416,Centro,Sistema de desempeño,2026-07-26
3,4,EMP001,2026-Q2,2026-06-30,71.6,79.8,78.1,93.3,9,8,...,El Salvador,San Salvador,San Salvador Centro,San Salvador,"Centro de San Salvador, San Salvador, El Salvador",13.681363,-89.226416,Centro,Sistema de desempeño,2026-07-26
4,5,EMP002,2025-Q3,2025-09-30,76.5,80.2,86.3,98.7,4,5,...,El Salvador,San Salvador,San Salvador Centro,San Salvador,"Centro de San Salvador, San Salvador, El Salvador",13.671427,-89.204773,Centro,Sistema de desempeño,2026-07-26


In [10]:
df_encuestas.head()

,encuesta_id,empleado_id,fecha_encuesta,canal,comentario_abierto,sentimiento_declarado,anonimizada,respuestas_satisfaccion_laboral,respuestas_relacion_con_jefatura,respuestas_reconocimiento,...,respuestas_intencion_de_permanecer,ubicacion_laboral_sede_codigo,ubicacion_laboral_pais,ubicacion_laboral_departamento,ubicacion_laboral_municipio,ubicacion_laboral_distrito,ubicacion_laboral_direccion_referencia,ubicacion_laboral_latitud,ubicacion_laboral_longitud,ubicacion_laboral_zona
0,ENC0001,EMP001,2026-06-04,Aplicación móvil,El ambiente es aceptable; espero más oportunid...,neutral,True,4,2,3,...,3,SED001,El Salvador,San Salvador,San Salvador Centro,San Salvador,"Centro de San Salvador, San Salvador, El Salvador",13.681363,-89.226416,Centro
1,ENC0002,EMP002,2026-06-07,Aplicación móvil,El ambiente laboral facilita mi productividad ...,positivo,True,5,5,5,...,5,SED001,El Salvador,San Salvador,San Salvador Centro,San Salvador,"Centro de San Salvador, San Salvador, El Salvador",13.671427,-89.204773,Centro
2,ENC0003,EMP003,2026-06-13,Aplicación móvil,"Cumplo mis funciones, pero necesito mayor clar...",neutral,True,3,2,3,...,4,SED002,El Salvador,La Libertad,La Libertad Sur,Santa Tecla,"Centro de Santa Tecla, La Libertad, El Salvador",13.661610,-89.274635,Centro
3,ENC0004,EMP004,2026-06-19,Portal interno,No tengo claridad sobre mis posibilidades de c...,negativo,True,1,3,2,...,1,SED002,El Salvador,La Libertad,La Libertad Sur,Santa Tecla,"Centro de Santa Tecla, La Libertad, El Salvador",13.675859,-89.273691,Centro
4,ENC0005,EMP005,2026-06-05,Correo,Tengo oportunidades de aprendizaje y deseo con...,positivo,True,4,4,5,...,4,SED003,El Salvador,Santa Ana,Santa Ana Centro,Santa Ana,"Centro de Santa Ana, Santa Ana, El Salvador",14.012323,-89.568194,Occidente


In [11]:
df_empleados.head()

,empleado_id,nombre_completo,genero,edad,fecha_ingreso,salario_mensual,modalidad,estado_laboral,fecha_salida,puesto,nivel_puesto,departamento,ubicacion_departamento
0,EMP001,Ana Hernández,F,29,2022-09-17,1400.0,Presencial,Activo,NaT,Desarrollador/a de Software,Operativo,Tecnología,San Salvador
1,EMP002,Carlos Vásquez,M,36,2025-07-26,1450.0,Remota,Activo,NaT,Analista de Datos,Operativo,Tecnología,San Salvador
2,EMP003,María Jiménez,F,53,2024-04-01,1100.0,Presencial,Activo,NaT,Analista de Recursos Humanos,Operativo,Recursos Humanos,San Salvador
3,EMP004,José Castro,M,29,2021-02-15,1650.0,Remota,Activo,NaT,Jefe/a de Recursos Humanos,Jefatura,Recursos Humanos,San Salvador
4,EMP005,Sofía Méndez,F,53,2022-04-13,1200.0,Híbrida,Activo,NaT,Analista Financiero/a,Operativo,Finanzas,Santa Tecla


## 4. Integrar las fuentes en un dataset consolidado

In [12]:
# empleado_id es la llave que conecta las 3 fuentes
df_consolidado = df_desempeno.merge(df_empleados, on="empleado_id", how="left")
df_consolidado = df_consolidado.merge(
    df_encuestas, on="empleado_id", how="left", suffixes=("", "_encuesta")
)

print(f"Dataset consolidado: {df_consolidado.shape}")
df_consolidado.head()

Dataset consolidado: (177, 57)


,registro_id,empleado_id,periodo,fecha_evaluacion,productividad_pct,calidad_trabajo_pct,cumplimiento_objetivos_pct,asistencia_pct,horas_capacitacion,horas_extra,...,respuestas_intencion_de_permanecer,ubicacion_laboral_sede_codigo,ubicacion_laboral_pais,ubicacion_laboral_departamento,ubicacion_laboral_municipio,ubicacion_laboral_distrito,ubicacion_laboral_direccion_referencia,ubicacion_laboral_latitud,ubicacion_laboral_longitud,ubicacion_laboral_zona
0,1,EMP001,2025-Q3,2025-09-30,76.2,72.1,68.6,97.6,12,9,...,3,SED001,El Salvador,San Salvador,San Salvador Centro,San Salvador,"Centro de San Salvador, San Salvador, El Salvador",13.681363,-89.226416,Centro
1,2,EMP001,2025-Q4,2025-12-31,56.2,76.7,71.5,94.1,11,12,...,3,SED001,El Salvador,San Salvador,San Salvador Centro,San Salvador,"Centro de San Salvador, San Salvador, El Salvador",13.681363,-89.226416,Centro
2,3,EMP001,2026-Q1,2026-03-31,75.1,67.9,76.0,92.8,10,19,...,3,SED001,El Salvador,San Salvador,San Salvador Centro,San Salvador,"Centro de San Salvador, San Salvador, El Salvador",13.681363,-89.226416,Centro
3,4,EMP001,2026-Q2,2026-06-30,71.6,79.8,78.1,93.3,9,8,...,3,SED001,El Salvador,San Salvador,San Salvador Centro,San Salvador,"Centro de San Salvador, San Salvador, El Salvador",13.681363,-89.226416,Centro
4,5,EMP002,2025-Q3,2025-09-30,76.5,80.2,86.3,98.7,4,5,...,5,SED001,El Salvador,San Salvador,San Salvador Centro,San Salvador,"Centro de San Salvador, San Salvador, El Salvador",13.671427,-89.204773,Centro


## 5. Calcular tres indicadores iniciales

In [13]:
productividad_promedio = round(df_consolidado["productividad_pct"].mean(), 2)

total_empleados = df_empleados["empleado_id"].nunique()
renunciaron = (df_empleados["estado_laboral"] == "Renunció").sum()
tasa_rotacion = round((renunciaron / total_empleados) * 100, 2)

satisfaccion_promedio = round(df_consolidado["respuestas_satisfaccion_laboral"].mean(), 2)

print("="*60)
print("INDICADORES INICIALES")
print("="*60)
print(f"1. Productividad promedio:        {productividad_promedio}%")
print(f"2. Tasa de rotación:               {tasa_rotacion}%  ({renunciaron} de {total_empleados} empleados)")
print(f"3. Satisfacción laboral promedio:  {satisfaccion_promedio} / 5")

INDICADORES INICIALES
1. Productividad promedio:        78.53%
2. Tasa de rotación:               15.0%  (6 de 40 empleados)
3. Satisfacción laboral promedio:  3.76 / 5


## 6. Crear el inventario de fuentes

In [14]:
df_inventario = pd.DataFrame([
    {
        "nombre_fuente": "Desempeño histórico",
        "tipo": "Archivo plano",
        "formato": "CSV",
        "ruta": str(ruta_csv),
        "registros": len(df_desempeno),
        "columnas": df_desempeno.shape[1],
        "llave_integracion": "empleado_id",
        "descripcion": "Evaluaciones trimestrales de desempeño por empleado.",
    },
    {
        "nombre_fuente": "Encuestas de clima laboral",
        "tipo": "API (simulada)",
        "formato": "JSON",
        "ruta": str(ruta_json),
        "registros": len(df_encuestas),
        "columnas": df_encuestas.shape[1],
        "llave_integracion": "empleado_id",
        "descripcion": "Resultados de encuestas de clima laboral.",
    },
    {
        "nombre_fuente": "Empleados, puestos y departamentos",
        "tipo": "Base de datos relacional",
        "formato": "SQL (SQLite)",
        "ruta": str(ruta_db),
        "registros": len(df_empleados),
        "columnas": df_empleados.shape[1],
        "llave_integracion": "empleado_id",
        "descripcion": "Datos maestros de empleado, puesto y departamento.",
    },
])

ruta_inventario_csv = BASE_DIR / "datos" / "procesados" / "inventario_fuentes.csv"
df_inventario.to_csv(ruta_inventario_csv, index=False, encoding="utf-8")

print(f"Inventario de fuentes guardado en: {ruta_inventario_csv}")
df_inventario

Inventario de fuentes guardado en: ..\datos\procesados\inventario_fuentes.csv


,nombre_fuente,tipo,formato,ruta,registros,columnas,llave_integracion,descripcion
0,Desempeño histórico,Archivo plano,CSV,..\datos\fuentes\desempeno_historico.csv,168,25,empleado_id,Evaluaciones trimestrales de desempeño por emp...
1,Encuestas de clima laboral,API (simulada),JSON,..\datos\fuentes\encuestas_clima_api.json,42,21,empleado_id,Resultados de encuestas de clima laboral.
2,"Empleados, puestos y departamentos",Base de datos relacional,SQL (SQLite),..\datos\fuentes\hr_analytics.db,40,13,empleado_id,"Datos maestros de empleado, puesto y departame..."


## 7. Guardar el dataset procesado

In [15]:
ruta_salida_csv = BASE_DIR / "datos" / "procesados" / "dataset_consolidado.csv"
df_consolidado.to_csv(ruta_salida_csv, index=False, encoding="utf-8")

print(f"Dataset procesado guardado en: {ruta_salida_csv}")

Dataset procesado guardado en: ..\datos\procesados\dataset_consolidado.csv


## 8. Documentar la evidencia en la bitácora semanal

In [17]:
from datetime import date

bitacora = f"""# Bitácora semanal — Semana 01

**Equipo:** 8 — HRTech People Analytics
**Fecha:** {date.today().isoformat()}

## Actividades realizadas
- Se definió el nombre del proyecto, el problema empresarial, el sector y los usuarios beneficiados.
- Se creó la estructura de carpetas del proyecto.
- Se documentó el README.md.
- Se seleccionó el CSV inicial de desempeño histórico.
- Se creó la base de datos SQL (hr_analytics.db) con 3 tablas relacionadas (empleados, puestos, departamentos).
- Se simuló la fuente JSON tipo API de encuestas de clima laboral.
- Se leyeron las tres fuentes desde Python (pandas + sqlite3 + json).
- Se integraron las tres fuentes en un dataset consolidado ({df_consolidado.shape[0]} filas, {df_consolidado.shape[1]} columnas).
- Se generó el inventario de fuentes.
- Se guardó el dataset procesado en datos/procesados/dataset_consolidado.csv.

## Evidencia
- Ver notebook: notebooks/guia_01_fuentes_heterogeneas.ipynb
- Ver archivos generados en: datos/procesados/

## Hallazgos iniciales
- Productividad promedio: {productividad_promedio}%
- Tasa de rotación: {tasa_rotacion}% ({renunciaron} de {total_empleados} empleados)
- Satisfacción laboral promedio: {satisfaccion_promedio} / 5
"""

ruta_bitacora = BASE_DIR / "documentos" / "bitacora_semana_01.md"
with open(ruta_bitacora, "w", encoding="utf-8") as f:
    f.write(bitacora)

print(f"Bitácora guardada en: {ruta_bitacora}")

Bitácora guardada en: ..\documentos\bitacora_semana_01.md
